# Hello, topic — the ten-line version

Three verbs. `topic()` opens a store, `.add()` puts text in, `.ask()` gets the best
chunks back out. That is the whole library at its smallest.

**Needs:** Ollama running with `nomic-embed-text` pulled — `ollama pull nomic-embed-text`.

**Note the import.** There is no `sys.path` hack in this notebook. The package is
installed (`pip install -e .` from the repo root), so this is the same import a
stranger gets from `pip install slim-llm-memory`.

In [1]:
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

from slim_llm_memory import topic

t = topic("hello", path=ROOT / ".hello_nb" / "topic")
t.add({
    "nginx.md": "To serve a site with nginx, install the package, put a server block in "
                "/etc/nginx/sites-enabled, then run `nginx -s reload` to pick it up.",
    "carbonara.md": "Carbonara is guanciale, pecorino romano, egg yolks and black pepper. "
                    "No cream, and the pan comes off the heat before the egg goes in.",
    "backups.md": "Backups run nightly at 03:00, are encrypted with age, and are kept for "
                  "30 days before rotation deletes them.",
})

added 3 doc(s), 3 chunks: 3 embedded, 0 unchanged, 0 removed

`add` reports what it did. Re-run the cell: the second time everything comes back
`unchanged`. Text is content-hashed, so unchanged text is never embedded twice — that is
what makes re-indexing a folder cheap.

In [2]:
r = t.ask("how do I reload the web server config?")
r

ask('how do I reload the web server config?')  3 hit(s) · hybrid · embed 2090 ms · scan 3.33 ms
   1  0.62  nginx.md#0               To serve a site with nginx, install the package, put a server   [both]
   2  0.45  backups.md#0             Backups run nightly at 03:00, are encrypted with age, and are   [dense]
   3  0.35  carbonara.md#0           Carbonara is guanciale, pecorino romano, egg yolks and black p  [dense]

One embedding call for the question, then a numpy scan over the stored vectors.
The two timings are reported separately because only one of them is your own code.

`r.context` is the block you hand to an LLM — numbered chunks, so the model can cite them.

In [3]:
print(r.context)

Context (retrieved for this prompt; cite by [n]):

[1] (nginx.md, score 0.62)
To serve a site with nginx, install the package, put a server block in /etc/nginx/sites-enabled, then run `nginx -s reload` to pick it up.

[2] (backups.md, score 0.45)
Backups run nightly at 03:00, are encrypted with age, and are kept for 30 days before rotation deletes them.

[3] (carbonara.md, score 0.35)
Carbonara is guanciale, pecorino romano, egg yolks and black pepper. No cream, and the pan comes off the heat before the egg goes in.


That is the core loop. Next: **`01_hello_library`** puts several topics behind one
handle, and **`03_hello_answer`** hands this context to a local model.

In [4]:
t.close()